In [16]:
import re
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt

In [17]:
def clean_parse_file():
	input_file = "./steam-dataset/games.csv"
	output_file = "./steam-dataset/games_fixed.csv"



	pattern = re.compile(r',(?=[^{}]*\})') 

	with open(input_file, "r", encoding="utf-8") as infile, \
		open(output_file, "w", encoding="utf-8") as outfile:

		for line in infile:
			line = pattern.sub("|", line)
			outfile.write(line)


df_games = pd.read_csv("./steam-dataset/games_fixed_clean.csv", encoding="ISO-8859-1", na_values="\\N")


def see_nan_values(df):
	nan_values = df.isnull().sum()


df_games.dropna(subset=["price_overview"], inplace=True)

nan_values = df_games.isnull().sum()


s = df_games["price_overview"].astype("string")

df_games["price"] = pd.to_numeric(
	s.str.extract(r'final\\?"?\s*[:|]\s*(\d+(?:\.\d+)?)', expand=False)
) / 100

df_games["currency"] = s.str.extract(
	r'currency\\?"?\s*[:|]\s*\\?"?([A-Z]{3})',
	expand=False
)

df_games

df_games.drop(columns=["price_overview", "languages", "type"], inplace=True)


unique_curr = df_games['currency'].value_counts()



from currency_converter import CurrencyConverter
c = CurrencyConverter('./eurofxref-hist.csv')

df_games = df_games.replace('SAR', 'ZAR')

rates = {
	currency: c.convert(1, currency, 'EUR')
	for currency in c.currencies
}

rates['PEN'] = 0.26
rates['UAH'] = 0.019
rates['COP'] = 0.00028
rates['KWD'] = 2.78
rates['KZT'] = 0.0019
rates['TWD'] = 0.027
rates['AED'] = 0.23
rates['VND'] = 0.000033
df_games['prices_eur'] = df_games['price'] * df_games['currency'].map(rates)


df_games.drop(columns=["price", "currency", 'is_free'])

# df_games[df_games["prices_eur"] < df_games["prices_eur"].quantile(0.99)]['prices_eur'].hist()


def one_hot_encoding(df, column): 

	for category in df[column].unique():
		df[category] = df[column].apply(lambda x: 1 if category in x else 0)


	return df.drop(column, axis=1)

df_categories = pd.read_csv('./steam-dataset/categories.csv', na_values="\\N")
df_genres = pd.read_csv("./steam-dataset/genres.csv", na_values="\\N")
df_reviews = pd.read_csv("./steam-dataset/reviews.csv", na_values="\\N")
df_insights = pd.read_csv("./steam-dataset/steamspy_insights.csv", encoding="ISO-8859-1", na_values="\\N")
df_tags = pd.read_csv("./steam-dataset/tags.csv", na_values="\\N")



df_genres["genre"].value_counts()

df_reviews.dropna(thresh=1)
df_reviews_important = df_reviews.drop(['metacritic_score', 'reviews', 'recommendations', 'steamspy_user_score', 'steamspy_score_rank', 'steamspy_positive', 'steamspy_negative'], axis=1)

categories = df_reviews_important['review_score_description'].unique()
df_reviews_important.dropna(subset=['review_score_description'], inplace=True)
categories = df_reviews_important['review_score_description'].unique()


df_insights_clean = df_insights.drop(['developer', 'publisher', 'price', 'initial_price', 'discount', 'languages', 'genres', 'playtime_average_forever', 'playtime_average_2weeks', 'playtime_median_forever', 'playtime_median_2weeks'], axis=1)
df_insights_clean

df_tags_count = df_tags["tag"].value_counts()

df_games = df_games.drop(["price", "currency", "is_free"], axis=1)

games = df_games.merge(df_reviews_important.set_index("app_id"), on="app_id", how="inner")
games = games.merge(df_insights_clean.set_index("app_id"), on="app_id", how="inner")
games

def standardize_data(df):

	'''
	This function standardize an array, its substracts mean value,
	and then divide the standard deviation.

	param 1: array
	return: standardized array
	'''

	mean = df.mean()
	std = df.std()
	new_df = (df - mean) / std

	return new_df, mean, std

games_one_hot = one_hot_encoding(games, 'owners_range')
games_one_hot = one_hot_encoding(games_one_hot, 'review_score_description')

games_final = games_one_hot.drop(columns=['app_id', 'name', 'release_date'])
games_final = games_final.sample(frac=1)

In [47]:
cols_x_ordenadas = [
    "price_eur",
    "review_score",
    "positive",
    "negative",
    "total",
    "concurrent_users_yesterday",
    "0 .. 20,000",
    "20,000 .. 50,000",
    "50,000 .. 100,000",
    "100,000 .. 200,000",
    "200,000 .. 500,000",
    "500,000 .. 1,000,000",
    "1,000,000 .. 2,000,000",
    "2,000,000 .. 5,000,000",
    "5,000,000 .. 10,000,000",
    "10,000,000 .. 20,000,000",
    "20,000,000 .. 50,000,000",
    "50,000,000 .. 100,000,000",
]

cols_y_ordenadas = [
    "Overwhelmingly Positive",
    "Very Positive",
    "Mostly Positive",
    "Positive",
    "Mixed",
    "Mostly Negative",
    "Negative",
    "Very Negative",
    "Overwhelmingly Negative",
    "No user reviews",
]

df_x = games_final.reindex(columns=cols_x_ordenadas)
df_y = games_final.reindex(columns=cols_y_ordenadas)

train_length = round(len(df_x) * .8)
df_x_train = df_x[:train_length]
df_x_test = df_x[train_length:]

df_y_train = df_y[:train_length]
df_y_test = df_y[train_length:]

df_x_standar, mean, std = standardize_data(df_x_train.iloc[:, :5])
df_x_train = pd.concat([df_x_standar.iloc[:, :], df_x_train.iloc[:, 5:]], axis=1)
df_x_train

df_x_test_standar = (df_x_test.iloc[:, :5] - mean) / std
df_x_test = pd.concat([df_x_test_standar, df_x_test.iloc[:, 5:]], axis=1)

display(df_x_train)
display(df_y_train)

,price_eur,review_score,positive,negative,total,concurrent_users_yesterday,"0 .. 20,000","20,000 .. 50,000","50,000 .. 100,000","100,000 .. 200,000","200,000 .. 500,000","500,000 .. 1,000,000","1,000,000 .. 2,000,000","2,000,000 .. 5,000,000","5,000,000 .. 10,000,000","10,000,000 .. 20,000,000","20,000,000 .. 50,000,000","50,000,000 .. 100,000,000"
43412,NaN,0.863093,-0.068876,-0.072369,-0.071284,0,1,0,0,0,0,0,0,0,0,0,0,0
30621,NaN,0.863093,-0.069436,-0.073349,-0.071909,0,1,0,0,0,0,0,0,0,0,0,0,0
21710,NaN,1.166332,-0.056193,-0.067473,-0.059135,1,1,0,0,0,0,0,0,0,0,0,0,0
19539,NaN,1.166332,-0.066887,-0.069432,-0.069127,0,0,1,0,0,0,0,0,0,0,0,0,0
45942,NaN,0.863093,-0.069187,-0.073349,-0.071682,0,1,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58938,NaN,0.863093,-0.069001,-0.072859,-0.071455,0,1,0,0,0,0,0,0,0,0,0,0,0
17793,NaN,0.256614,-0.069436,-0.071390,-0.071682,0,1,0,0,0,0,0,0,0,0,0,0,0
43007,NaN,0.863093,-0.068752,-0.073349,-0.071284,0,1,0,0,0,0,0,0,0,0,0,0,0
41980,NaN,1.166332,0.017791,0.002545,0.016541,9,0,0,0,1,0,0,0,0,0,0,0,0


,Overwhelmingly Positive,Very Positive,Mostly Positive,Positive,Mixed,Mostly Negative,Negative,Very Negative,Overwhelmingly Negative,No user reviews
43412,0,0,0,1,0,0,0,0,0,0
30621,0,0,0,1,0,0,0,0,0,0
21710,0,1,0,1,0,0,0,0,0,0
19539,0,1,0,1,0,0,0,0,0,0
45942,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
58938,0,0,0,1,0,0,0,0,0,0
17793,0,0,0,0,1,0,0,0,0,0
43007,0,0,0,1,0,0,0,0,0,0
41980,0,1,0,1,0,0,0,0,0,0


In [48]:
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

In [49]:
rnd_clf = RandomForestClassifier(n_estimators=400, max_leaf_nodes=10, n_jobs=-1, random_state=42)
rnd_clf.fit(df_x_train, df_y_train)
y_pred_rf= rnd_clf.predict(df_x_test)
print("Accuracy", accuracy_score(df_y_test, y_pred_rf))
rnd_clf

Accuracy 0.8361718852351631


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",400
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total n

In [ ]:
y_pred_raw = rnd_clf.predict(df_x_test[:10])
y_test_raw = df_y_test.iloc[:10].to_numpy()

class_names = df_y_test.columns.tolist()

actual_indices = np.argmax(y_test_raw, axis=1)
predicted_indices = np.argmax(y_pred_raw, axis=1)

actual_labels = [class_names[idx] for idx in actual_indices]
predicted_labels = [class_names[idx] for idx in predicted_indices]

results_rf = pd.DataFrame(
    {
        "Clase Esperada": actual_labels,
        "Clase Predicha": predicted_labels,
        "¿Correcto?": np.array(actual_labels) == np.array(predicted_labels),
    }
)

display(results_rf)

,Clase Esperada,Clase Predicha,¿Correcto?
0,Mostly Negative,Overwhelmingly Positive,False
1,Mixed,Mixed,True
2,Positive,Positive,True
3,Positive,Positive,True
4,Very Positive,Very Positive,True
5,No user reviews,No user reviews,True
6,Very Positive,Very Positive,True
7,No user reviews,No user reviews,True
8,No user reviews,No user reviews,True
9,Mostly Positive,Positive,False


In [53]:
def capturar_datos_juego():
    print("=== CAPTURA DE DATOS DEL JUEGO DE STEAM ===")

    price_eur = float(input("Precio en EUR (ej. 19.99): "))
    review_score = float(input("Calificación (ej. 8.4): "))
    positive = int(input("Cantidad de reseñas positivas (ej. 15000): "))
    negative = int(input("Cantidad de reseñas negativas (ej. 1200): "))
    total = positive + negative  # Cálculo directo del total
    concurrent_users = int(input("Usuarios concurrentes ayer (ej. 4500): "))

    rangos_owners = [
        "0 .. 20,000",
        "20,000 .. 50,000",
        "50,000 .. 100,000",
        "100,000 .. 200,000",
        "200,000 .. 500,000",
        "500,000 .. 1,000,000",
        "1,000,000 .. 2,000,000",
        "2,000,000 .. 5,000,000",
        "5,000,000 .. 10,000,000",
        "10,000,000 .. 20,000,000",
        "20,000,000 .. 50,000,000",
        "50,000,000 .. 100,000,000",
    ]

    # Corregida la coma faltante en 'Very Negative'
    # grade = [
    #     "Overwhelmingly Positive",
    #     "Very Positive",
    #     "Mostly Positive",
    #     "Positive",
    #     "Mixed",
    #     "Negative",
    #     "Mostly Negative",
    #     "Very Negative",
    #     "Overwhelmingly Negative",
    #     "No user reviews",
    # ]

    print("\n--- Selecciona el rango de propietarios (Owners Range) ---")
    for i, rango in enumerate(rangos_owners, 1):
        print(f"[{i}] {rango}")

    while True:
        try:
            opcion = int(input(f"Elige una opción (1-{len(rangos_owners)}): "))
            if 1 <= opcion <= len(rangos_owners):
                owners_selected_idx = opcion - 1
                break
            else:
                print("Opción fuera de rango. Intenta de nuevo.")
        except ValueError:
            print("Por favor, ingresa solo un número entero.")

    # print("\n--- Selecciona la calificación de acuerdo a reseñas ---")
    # for i, rango in enumerate(grade, 1):
    #     print(f"[{i}] {rango}")

    # while True:
    #     try:
    #         opcion = int(input(f"Elige una opción (1-{len(grade)}): "))
    #         if 1 <= opcion <= len(grade):
    #             grade_selected_idx = opcion - 1
    #             break
    #         else:
    #             print("Opción fuera de rango. Intenta de nuevo.")
    #     except ValueError:
    #         print("Por favor, ingresa solo un número entero.")

    datos_capturados = {
        "price_eur": price_eur,
        "review_score": review_score,
        "positive": positive,
        "negative": negative,
        "total": total,
        "concurrent_users_yesterday": concurrent_users,
    }

    for idx, rango in enumerate(rangos_owners):
        column_name = rango
        datos_capturados[column_name] = 1 if idx == owners_selected_idx else 0

    # for idx, g in enumerate(grade):
    #     column_name = f"grade_{g}"
    #     datos_capturados[column_name] = 1 if idx == grade_selected_idx else 0

    print("\n¡Datos capturados y procesados exitosamente!")
    return pd.DataFrame([datos_capturados])

In [55]:
entradas = capturar_datos_juego()
print(entradas)
prediction = rnd_clf.predict(entradas)

class_names = df_y_test.columns.tolist()
predicted_indices = np.argmax(prediction, axis=1)
predicted_labels = [class_names[idx] for idx in predicted_indices]

print("Valor predecido: ", predicted_labels)

=== CAPTURA DE DATOS DEL JUEGO DE STEAM ===

--- Selecciona el rango de propietarios (Owners Range) ---
[1] 0 .. 20,000
[2] 20,000 .. 50,000
[3] 50,000 .. 100,000
[4] 100,000 .. 200,000
[5] 200,000 .. 500,000
[6] 500,000 .. 1,000,000
[7] 1,000,000 .. 2,000,000
[8] 2,000,000 .. 5,000,000
[9] 5,000,000 .. 10,000,000
[10] 10,000,000 .. 20,000,000
[11] 20,000,000 .. 50,000,000
[12] 50,000,000 .. 100,000,000

¡Datos capturados y procesados exitosamente!
   price_eur  review_score  positive  negative   total  \
0      19.99           8.4    150000      1200  151200   

   concurrent_users_yesterday  0 .. 20,000  20,000 .. 50,000  \
0                        4500            0                 0   

   50,000 .. 100,000  100,000 .. 200,000  200,000 .. 500,000  \
0                  0                   0                   0   

   500,000 .. 1,000,000  1,000,000 .. 2,000,000  2,000,000 .. 5,000,000  \
0                     0                       0                       1   

   5,000,000 .. 10,00